# 04 — Final Neuromorphic Student: Architecture, Distillation & Training**Purpose:** Train the single deployable model used by the exhibition — the **Improved Student** — from scratch inside this notebook, with **per-epoch logging** for every training run.## Final architecture`10 × 30 s PSG epochs → Lite Multi-Resolution Stem → Depthwise-Separable CNN → Parametric Gabor FEB → 2-Layer GRU → 5-Class classifier`Training is a two-stage process, fully executed in this notebook:1. **Stage 1 — Improved Teacher:** trained from scratch with focal loss + class weights, per-epoch validation logging.2. **Stage 2 — Improved Student:** distilled from the teacher (hard-label CE + softened KL + feature alignment), per-epoch validation logging.Knowledge distillation transfers information from a trained teacher into the smaller student. The teacher is a training-time component only; the student checkpoint is the only model exported for deployment and exhibition.### Final configuration- Sequence length: **10 epochs** (300 s context)- Input channels: **4** | Classes: **5**- Training budget: **20 epochs per stage**- Checkpoints: `artifacts/exhibition/EXP-EXHIBITION-15SUBJ/seed-42/teacher_improved_best.pt`, `artifacts/exhibition/EXP-EXHIBITION-15SUBJ/seed-42/student_best.pt`

In [ ]:
import mathimport timeimport randomimport subprocessfrom pathlib import Pathimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport torchimport torch.nn as nnimport torch.nn.functional as Ffrom torch.utils.data import Dataset, DataLoaderfrom torch.optim import AdamWfrom torch.optim.lr_scheduler import LambdaLRfrom sklearn.metrics import cohen_kappa_score, accuracy_score, f1_scoreSEED = 42SEQ_LEN = 10SEQ_STRIDE = 5EPOCHS = 20BATCH_SIZE = 16LEARNING_RATE = 3e-4WEIGHT_DECAY = 1e-4GRAD_CLIP = 1.0N_CHANNELS = 4N_CLASSES = 5FS = 100DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")PROJECT_ROOT = Path("/home/shamique/projects/sleep")CACHE_DIR = PROJECT_ROOT / "data/cache"ARTIFACT_DIR = PROJECT_ROOT / "artifacts"RESULTS_DIR = PROJECT_ROOT / "results"ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)RESULTS_DIR.mkdir(parents=True, exist_ok=True)TEACHER_CHECKPOINT = ARTIFACT_DIR / "exhibition" / "EXP-EXHIBITION-15SUBJ" / "seed-42" / "teacher_improved_best.pt"STUDENT_CHECKPOINT = ARTIFACT_DIR / "exhibition" / "EXP-EXHIBITION-15SUBJ" / "seed-42" / "student_best.pt"STUDENT_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)random.seed(SEED)np.random.seed(SEED)torch.manual_seed(SEED)if torch.cuda.is_available():    torch.cuda.manual_seed_all(SEED)print("Device:", DEVICE)print("Teacher checkpoint:", TEACHER_CHECKPOINT)print("Student checkpoint:", STUDENT_CHECKPOINT)

## 1. Sequence datasetEach training example contains ten contiguous 30-second epochs. Windows are formed independently inside each subject-night cache so no temporal sequence crosses recording boundaries, and the subject-level split from Notebook 01 is respected (no subject leakage).

In [ ]:
class SleepSequenceDataset(Dataset):    def __init__(self, cache_index_df, split, seq_len=SEQ_LEN, stride=SEQ_STRIDE):        self.samples = []        self.cache = {}        self.seq_len = seq_len        rows = cache_index_df.loc[cache_index_df["split"] == split]        for _, row in rows.iterrows():            path = row["cache_path"]            data = np.load(path)            n = len(data["labels"])            for start in range(0, n - seq_len + 1, stride):                self.samples.append((path, start))    def _load(self, path):        if path not in self.cache:            d = np.load(path)            self.cache[path] = (d["epochs"], d["labels"])        return self.cache[path]    def __len__(self):        return len(self.samples)    def __getitem__(self, index):        path, start = self.samples[index]        epochs, labels = self._load(path)        end = min(start + self.seq_len, len(labels))        x = epochs[start:end]        y = labels[start:end]        if len(x) < self.seq_len:            pad = self.seq_len - len(x)            x = np.concatenate([x, np.repeat(x[-1:], pad, axis=0)], axis=0)            y = np.concatenate([y, np.repeat(y[-1:], pad, axis=0)], axis=0)        # Verify temporal continuity using orig_epoch_idx if available        data = np.load(path)        if "orig_epoch_idx" in data:            orig_idx = data["orig_epoch_idx"]            window_idx = orig_idx[start:end]            if len(window_idx) > 1 and not np.all(np.diff(window_idx) == 1):                raise ValueError(f"Temporal gap in window for {path} at start={start}")        return torch.from_numpy(x).float(), torch.from_numpy(y).long()cache_index = pd.read_csv(CACHE_DIR / "cache_index.csv")import json# Use exhibition manifest as single source of split truthwith open('../data/manifests/exhibition_15subj_v1.json') as f:    exhibition_manifest = json.load(f)train_subjects = exhibition_manifest['train_subjects']val_subjects = exhibition_manifest['validation_subjects']test_subjects = exhibition_manifest['test_subjects']# Filter cache index by manifest subjectstrain_cache = cache_index[cache_index['subject_id'].isin(train_subjects)]val_cache = cache_index[cache_index['subject_id'].isin(val_subjects)]test_cache = cache_index[cache_index['subject_id'].isin(test_subjects)]# Verify no overlapassert set(train_subjects).isdisjoint(val_subjects)assert set(train_subjects).isdisjoint(test_subjects)assert set(val_subjects).isdisjoint(test_subjects)train_ds = SleepSequenceDataset(train_cache, "train")val_ds = SleepSequenceDataset(val_cache, "val")test_ds = SleepSequenceDataset(test_cache, "test")train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)print("Sequence windows:")print("train:", len(train_ds))print("val:  ", len(val_ds))print("test: ", len(test_ds))

## 2. Model architecturesThe canonical implementations live in `src/sleep_staging/models/` and are imported here.**Improved Teacher:** multi-resolution stem → residual encoder → learned spectral filterbank → gated fusion → 2-layer transformer encoder.**Improved Student (final deployable model):** Lite multi-resolution stem → depthwise-separable CNN → parametric Gabor FEB → 2-layer GRU.

In [ ]:
import syssys.path.insert(0, str(PROJECT_ROOT / "src"))from sleep_staging.models import ImprovedTeacher, ImprovedStudentteacher = ImprovedTeacher().to(DEVICE)student = ImprovedStudent().to(DEVICE)n_teacher = sum(p.numel() for p in teacher.parameters())n_student = sum(p.numel() for p in student.parameters())print(f"Improved Teacher parameters: {n_teacher:,}")print(f"Improved Student parameters: {n_student:,}  (final deployable model)")with torch.no_grad():    smoke = torch.randn(2, SEQ_LEN, N_CHANNELS, 30 * FS, device=DEVICE)    t_logits = teacher(smoke)    s_logits, s_feat = student(smoke, return_features=True)print("Teacher logits:", tuple(t_logits.shape))print("Student logits:", tuple(s_logits.shape), "| features:", tuple(s_feat.shape))

## 3. Class weighting from the training partitionClass weights are computed from training labels only. The validation and test partitions are never used to determine training weights.

In [ ]:
train_labels = []for _, row in cache_index.loc[cache_index["split"] == "train"].iterrows():    train_labels.append(np.load(row["cache_path"])["labels"])train_labels = np.concatenate(train_labels)class_counts = np.bincount(train_labels, minlength=N_CLASSES)counts = torch.tensor(class_counts, dtype=torch.float32, device=DEVICE)class_weights = torch.log(counts.sum() / (counts + 1.0))class_weights = class_weights / class_weights.mean()print("Training class counts:", class_counts.tolist())print("Class weights:", [round(float(w), 3) for w in class_weights])

## 4. Training objectives**Teacher — Focal loss** (imbalance-aware supervision):**Student — Distillation loss:** hard-label cross-entropy + temperature-scaled teacher-student KL divergence, exactly as in the canonical pipeline.

In [ ]:
class FocalLoss(nn.Module):    def __init__(self, weight, gamma=1.5):        super().__init__()        self.w = weight        self.gamma = gamma    def forward(self, logits, labels):        C = logits.shape[-1]        ce = F.cross_entropy(            logits.reshape(-1, C), labels.reshape(-1),            weight=self.w.to(logits.device), reduction="none",        )        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()class DistillLoss(nn.Module):    def __init__(self, weight, T_s=8, T_e=4, epochs=EPOCHS):        super().__init__()        self.ce = nn.CrossEntropyLoss(weight=weight)        self.Ts, self.Te, self.epochs = T_s, T_e, epochs    def temp(self, ep):        return self.Ts + (self.Te - self.Ts) * min(ep / self.epochs, 1)    def forward(self, s_logits, t_logits, labels, ep):        C = s_logits.shape[-1]        sl = s_logits.reshape(-1, C)        tl = t_logits.reshape(-1, C).detach()        lb = labels.reshape(-1)        ce = self.ce(sl, lb)        tmp = self.temp(ep)        kl = F.kl_div(            F.log_softmax(sl / tmp, -1),            F.softmax(tl / tmp, -1),            reduction="batchmean",        ) * (tmp ** 2)        return ce + 0.5 * klfocal_loss = FocalLoss(class_weights)distill_loss = DistillLoss(class_weights)print("FocalLoss and DistillLoss ready.")

## 5. Training & evaluation utilities (per-epoch logging)Every training loop below logs **every epoch** — train loss, validation Cohen's κ, validation accuracy, and macro-F1 — and checkpoints only the best model according to validation κ.

In [ ]:
@torch.no_grad()def collect_predictions(model, loader):    model.eval()    y_true, y_pred = [], []    for x, y in loader:        logits = model(x.to(DEVICE))        pred = logits.argmax(dim=-1).cpu().numpy()        y_true.append(y.numpy().reshape(-1))        y_pred.append(pred.reshape(-1))    return np.concatenate(y_true), np.concatenate(y_pred)def evaluate(model, loader, tag, epoch, total_epochs, train_loss):    y_true, y_pred = collect_predictions(model, loader)    kappa = cohen_kappa_score(y_true, y_pred)    acc = accuracy_score(y_true, y_pred)    mf1 = f1_score(y_true, y_pred, average="macro", zero_division=0)    print(        f"  [{tag}] Epoch {epoch:02d}/{total_epochs:02d} | "        f"train_loss={train_loss:.4f} | val_kappa={kappa:.4f} | "        f"val_acc={acc:.4f} | val_macroF1={mf1:.4f}"    )    return {"kappa": kappa, "acc": acc, "mf1": mf1}def cosine_warmup(optimizer, total_steps):    def lr_lambda(s):        if s < 0.1 * total_steps:            return s / (0.1 * total_steps)        return 0.5 * (1 + math.cos(math.pi * (s - 0.1 * total_steps) / (0.9 * total_steps)))    return LambdaLR(optimizer, lr_lambda)def train_model(model, loss_fn, train_ld, val_ld, epochs, tag, ckpt_path,                teacher=None,                train_subjects=None,                val_subjects=None,                test_subjects=None):    opt = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)    sched = cosine_warmup(opt, epochs * len(train_ld))    best_kappa = -1.0    history = []    if teacher is not None:        teacher.eval()    for ep in range(epochs):        model.train()        loss_sum, n_batches = 0.0, 0        for x, y in train_ld:            x, y = x.to(DEVICE), y.to(DEVICE)            opt.zero_grad(set_to_none=True)            if teacher is not None:                with torch.no_grad():                    t_logits = teacher(x)                loss = loss_fn(model(x), t_logits, y, ep)            else:                loss = loss_fn(model(x), y)            loss.backward()            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)            opt.step()            sched.step()            loss_sum += float(loss.detach())            n_batches += 1        metrics = evaluate(            model, val_ld, tag, ep + 1, epochs,            loss_sum / max(n_batches, 1),        )        history.append({"epoch": ep + 1, "train_loss": loss_sum / max(n_batches, 1), **metrics})        if metrics["kappa"] > best_kappa:            best_kappa = metrics["kappa"]            checkpoint = {                "model_state_dict": model.state_dict(),                "experiment_id": "EXP-EXHIBITION-15SUBJ",                "dataset": "Sleep-EDF Expanded",                "n_records": len(train_subjects) + len(val_subjects) + len(test_subjects),                "n_persons": len(set(train_subjects + val_subjects + test_subjects)),                "split_manifest": "data/manifests/exhibition_15subj_v1.json",                "split_type": "subject",                "seed": SEED,                "sequence_length": SEQ_LEN,                "training_stride": SEQ_STRIDE,                "evaluation_protocol": "legacy_all_position",                "train_subjects": train_subjects,                "validation_subjects": val_subjects,                "test_subjects": test_subjects,                "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip() if Path(".git").exists() else "unknown",                "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ"),            }            torch.save(checkpoint, ckpt_path)            print(f"    -> new best {tag} (kappa={best_kappa:.4f}); checkpoint saved.")    payload = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
            if model_state_dict" in payload:
                state = payload["model_state_dict

## 6. Stage 1 — Train the Improved Teacher (per-epoch logging)The teacher is trained from scratch on the training split with focal loss. Each epoch prints its full validation metrics; only the best-κ checkpoint is kept.

In [ ]:
print("Training Improved Teacher")print("=" * 80)t0 = time.time()teacher, teacher_history = train_model(    teacher, focal_loss, train_loader, val_loader, EPOCHS,    "teacher", TEACHER_CHECKPOINT,    train_subjects=train_subjects,    val_subjects=val_subjects,    test_subjects=test_subjects,)print(f"Teacher training finished in {time.time() - t0:.1f}s | best val kappa: {teacher_history['kappa'].max():.4f}")display(teacher_history)

## 7. Stage 2 — Distill into the Improved Student (per-epoch logging)The student learns from the teacher's softened logits plus the hard labels. This is the final deployable model of the project.

In [ ]:
print("Training Improved Student (distillation)")print("=" * 80)t0 = time.time()student, student_history = train_model(    student, distill_loss, train_loader, val_loader, EPOCHS,    "student", STUDENT_CHECKPOINT,    teacher=teacher,    train_subjects=train_subjects,    val_subjects=val_subjects,    test_subjects=test_subjects,)print(f"Student training finished in {time.time() - t0:.1f}s | best val kappa: {student_history['kappa'].max():.4f}")display(student_history[['epoch', 'train_loss', 'val_kappa', 'val_acc', 'val_macroF1']])

## 8. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))axes[0].plot(teacher_history["epoch"], teacher_history["train_loss"], label="Teacher")axes[0].plot(student_history["epoch"], student_history["train_loss"], label="Student")axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Train loss")axes[0].set_title("Training loss"); axes[0].legend(); axes[0].grid(alpha=0.3)axes[1].plot(teacher_history["epoch"], teacher_history["kappa"], label="Teacher", marker="o")axes[1].plot(student_history["epoch"], student_history["kappa"], label="Student", marker="s")axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation Cohen's κ")axes[1].set_title("Validation κ per epoch"); axes[1].legend(); axes[0].grid(alpha=0.3)axes[1].grid(alpha=0.3)plt.tight_layout()plt.show()

## Handoff to Notebook 05`artifacts/exhibition/EXP-EXHIBITION-15SUBJ/seed-42/student_best.pt` is the single deployable checkpoint. Notebook 05 is the only notebook that publishes the official test metrics on it. Training logs are diagnostic artifacts and must not be turned into alternative final-result tables.